In [2]:
from pyspark.sql import SparkSession
import sys,os 

In [3]:
spark = SparkSession.builder.appName("features").master("local[*]").getOrCreate()

In [4]:
print('Python',sys.executable)
print('Java',os.environ.get('JAVA_HOME'))
print('hadop',os.environ.get('HADOOP_HOME'))

Python C:\Users\aaron\anaconda3\python.exe
Java C:\Users\aaron\OneDrive\Escritorio\big_data\jdk-17.0.12
hadop C:\Users\aaron\OneDrive\Escritorio\big_data\winutils-master\hadoop-3.3.6


In [ ]:
df = spark.read.csv('data_proyecto_bigkk/train/train.csv', 
                    inferSchema=True, 
                    sep=',', 
                    header=True, 
                    multiLine=True,  
                    escape='"')      

df_clean = df.na.fill({'Description': ''})

In [37]:
df.show(5)

+----+-----------+---+------+------+------+------+------+------+------------+---------+----------+--------+----------+------+--------+---+-----+--------------------+--------+--------------------+---------+--------+-------------+
|Type|       Name|Age|Breed1|Breed2|Gender|Color1|Color2|Color3|MaturitySize|FurLength|Vaccinated|Dewormed|Sterilized|Health|Quantity|Fee|State|           RescuerID|VideoAmt|         Description|    PetID|PhotoAmt|AdoptionSpeed|
+----+-----------+---+------+------+------+------+------+------+------------+---------+----------+--------+----------+------+--------+---+-----+--------------------+--------+--------------------+---------+--------+-------------+
|   2|     Nibble|  3|   299|     0|     1|     1|     7|     0|           1|        1|         2|       2|         2|     1|       1|100|41326|8480853f516546f6c...|       0|Nibble is a 3+ mo...|86e1089a3|     1.0|            2|
|   2|No Name Yet|  1|   265|     0|     1|     1|     2|     0|           2|       

In [38]:
from pyspark.ml.feature import Word2Vec, Tokenizer
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

In [39]:
tokenizer = Tokenizer(inputCol="Description", outputCol="tokens")
wordsData = tokenizer.transform(df)

In [40]:
word2Vec = Word2Vec(vectorSize=20,minCount=1,windowSize=5,inputCol="tokens",outputCol="W2V")

In [41]:
model = word2Vec.fit(wordsData)

In [42]:
result = model.transform(wordsData)

In [43]:
def vector_to_string(vector):
    return str(vector.toArray().tolist())

vector_udf = udf(vector_to_string, StringType())

In [44]:
from pyspark.sql.functions import col, regexp_replace

In [45]:
df_con_texto = result.withColumn("W2V_String", col("W2V").cast("string"))

In [46]:
df_limpio = df_con_texto.withColumn("W2V_String", regexp_replace("W2V_String", "\\[|\\]", "")) \
                        .withColumn("W2V_String", regexp_replace("W2V_String", ",", ""))

In [47]:
final_df = df_limpio.select("PetID", "W2V_String")

In [34]:
final_df.write.csv("resultado_vectores_clean", header=True, mode="overwrite")

In [48]:
pandas_df = final_df.toPandas()
pandas_df.to_csv("feature_texto.csv", index=False)